# 10 Batch MD Benchmark

This notebook is a launcher and tracker for the multi-system MD benchmark for small PHA oligomers. The benchmark checks whether the existing P3HB_4 workflow also works for related PHA oligomers with different side chains and lengths.

Run one system at a time to avoid long blocking notebook runs. Larger oligomers may take much longer during GAFF2/antechamber, so the reusable automation stays in `scripts/run_md_test_set.py`.

For each selected system, the batch script follows the same staged shape as the existing `examples/output/md_tests/P3HB_4/` tutorial output: `gaff2/`, `openmm/dry_polymer/`, `gromacs/dry_polymer/`, and `gromacs/solvated_polymer/`.

## Benchmark Systems

The test set contains the six systems below.

In [ ]:
from iphasimulator.workflows.md_benchmark import SYSTEMS

systems = list(SYSTEMS)
systems

## Choose One Polymer

Edit only `selected_system`, then run the notebook from this cell downward.

The notebook automatically prepares, solvates, minimises, and submits only this polymer. Completed stages are skipped when you rerun it.

In [ ]:
selected_system = systems[-1]

if selected_system not in systems:
    raise ValueError(f"Unknown benchmark system: {selected_system}")

selected_system

## Output Folder Structure

Benchmark output is written under `examples/output/benchmark/`.

Expected system folders:

- `examples/output/benchmark/P3HB_4/`
- `examples/output/benchmark/P3HB_8/`
- `examples/output/benchmark/P3HO_4/`
- `examples/output/benchmark/P3HO_8/`
- `examples/output/benchmark/P3HDD_4/`
- `examples/output/benchmark/P3HDD_8/`

In [ ]:
from pathlib import Path

repo_root = Path.cwd() if (Path.cwd() / "src" / "iphasimulator").exists() else Path.cwd().parent
md_tests_root = repo_root / "examples" / "output" / "benchmark"
expected_system_dirs = [md_tests_root / system for system in systems]

print(f"Repository: {repo_root}")
print(f"Benchmark output root: {md_tests_root}")
for path in expected_system_dirs:
    print(path.relative_to(repo_root))

## Run the Selected Polymer Automatically

Run the cell below once. For `selected_system`, it automatically:

1. prepares missing benchmark/GROMACS workflow files;
2. runs `run_solvate_local.sh` when solvation is incomplete;
3. runs `run_step6_local.sh` when minimisation is incomplete;
4. submits `run_hpc_equilibration_production.slurm` automatically when `sbatch` is available.

If this notebook is running on a local computer without `sbatch`, it prints the HPC submission command instead. After submitting this polymer, change `selected_system` to the next polymer and rerun from the selection cell.

This HPC script has been tested on the KCL GPU cluster using `gromacs/2021.5-gcc-11.4.0-cuda-11.8.0`. Users on other HPC systems may need to modify the SLURM directives and module load command.

In [ ]:
import os
import shutil
import subprocess
import sys

REUSE_EXISTING_GAFF2 = True
STOP_ON_ERROR = True

src_path = repo_root / "src"
env = os.environ.copy()
env["PYTHONPATH"] = str(src_path)
selected_solvated_dir = md_tests_root / selected_system / "gromacs" / "solvated_polymer"

benchmark_cmd = [
    sys.executable,
    "-m",
    "iphasimulator.workflows.md_benchmark",
    "--system",
    selected_system,
    "--prepare-gromacs",
    "--skip-openmm",
]
if REUSE_EXISTING_GAFF2:
    benchmark_cmd.append("--reuse-existing-gaff2")
if STOP_ON_ERROR:
    benchmark_cmd.append("--stop-on-error")

print("Selected benchmark system:", selected_system)
print("Reuse existing GAFF2 outputs:", REUSE_EXISTING_GAFF2)

solvation_script = selected_solvated_dir / "run_solvate_local.sh"
minimization_script = selected_solvated_dir / "run_step6_local.sh"
neutralized_structure = selected_solvated_dir / "system_neutralized.gro"
minimized_structure = selected_solvated_dir / "step6.0_minimization.gro"
hpc_script = selected_solvated_dir / "run_hpc_equilibration_production.slurm"
submission_record = selected_solvated_dir / "hpc_submission.txt"

def run_selected_gromacs_script(script_name):
    script_path = selected_solvated_dir / script_name
    if not script_path.exists():
        raise FileNotFoundError(f"Missing {script_path}")
    print(f"[{selected_system}] bash {script_name}")
    return subprocess.run(
        ["bash", script_name],
        cwd=selected_solvated_dir,
        check=True,
    )

def read_primary_molecule_charge(topology_path):
    if not topology_path.exists():
        return None
    in_atoms = False
    total_charge = 0.0
    for line in topology_path.read_text().splitlines():
        stripped = line.strip()
        if stripped.startswith("["):
            if in_atoms:
                break
            in_atoms = stripped.lower() == "[ atoms ]"
            continue
        if in_atoms and stripped and not stripped.startswith(";"):
            fields = stripped.split(";", 1)[0].split()
            if len(fields) >= 7:
                total_charge += float(fields[6])
    return total_charge

dry_topology = md_tests_root / selected_system / "gromacs" / "dry_polymer" / "topol.top"
dry_topology_charge = read_primary_molecule_charge(dry_topology)
needs_charge_repair = (
    dry_topology_charge is not None
    and abs(dry_topology_charge) > 1e-4
)
if dry_topology_charge is not None:
    print(f"[{selected_system}] current dry GROMACS topology charge: {dry_topology_charge:.8f}")
if needs_charge_repair:
    print(f"[{selected_system}] rebuilding GROMACS files to repair non-integer topology charge")

required_workflow_files = [solvation_script, minimization_script, hpc_script]
workflow_reprepared = False
if needs_charge_repair or not all(path.exists() for path in required_workflow_files):
    print(f"[{selected_system}] preparing benchmark and GROMACS workflow files")
    print("Benchmark command:", " ".join(str(part) for part in benchmark_cmd))
    subprocess.run(benchmark_cmd, env=env, check=True)
    workflow_reprepared = True
    repaired_charge = read_primary_molecule_charge(dry_topology)
    if repaired_charge is None or abs(repaired_charge) > 1e-4:
        raise RuntimeError(f"GROMACS topology charge repair failed: {repaired_charge}")
    print(f"[{selected_system}] repaired dry GROMACS topology charge: {repaired_charge:.8f}")
else:
    print(f"[{selected_system}] workflow files already exist; skipping preparation")

neutralisation_outputs = [
    selected_solvated_dir / "system_solvated.gro",
    selected_solvated_dir / "ions.tpr",
    neutralized_structure,
    selected_solvated_dir / "step5_input.gro",
]
if workflow_reprepared:
    run_selected_gromacs_script("run_solvate_local.sh")
    missing = [path for path in neutralisation_outputs if not path.exists()]
    if missing:
        raise RuntimeError(f"Solvation did not produce required outputs: {missing}")
elif minimized_structure.exists():
    print(f"[{selected_system}] minimized structure exists; skipping solvation")
elif not all(path.exists() for path in neutralisation_outputs):
    run_selected_gromacs_script("run_solvate_local.sh")
    missing = [path for path in neutralisation_outputs if not path.exists()]
    if missing:
        raise RuntimeError(f"Solvation did not produce required outputs: {missing}")
else:
    print(f"[{selected_system}] solvation already complete; skipping run_solvate_local.sh")

if not minimized_structure.exists():
    run_selected_gromacs_script("run_step6_local.sh")
    if not minimized_structure.exists():
        raise RuntimeError(f"Minimization did not produce {minimized_structure}")
else:
    print(f"[{selected_system}] minimization already complete; skipping run_step6_local.sh")

if not hpc_script.exists():
    raise FileNotFoundError(f"Missing HPC submission script: {hpc_script}")
if submission_record.exists():
    print(f"[{selected_system}] HPC submission already recorded:")
    print(submission_record.read_text().strip())
elif shutil.which("sbatch"):
    submission = subprocess.run(
        ["sbatch", hpc_script.name],
        cwd=selected_solvated_dir,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=True,
    )
    submission_record.write_text(submission.stdout)
    print(f"[{selected_system}] {submission.stdout.strip()}")
else:
    print(f"[{selected_system}] ready for HPC. Submit on the cluster with:")
    print(f"  cd {selected_solvated_dir}")
    print("  sbatch run_hpc_equilibration_production.slurm")

selected_index = systems.index(selected_system)
if selected_index + 1 < len(systems):
    print(f'Next polymer: selected_system = "{systems[selected_index + 1]}"')
else:
    print("This was the final polymer in the benchmark list.")

## Check Progress

These checks report progress across all six benchmark systems, including neutralisation, local minimisation readiness, and the generated HPC submission script.

In [ ]:
folder_status = {
    system: (md_tests_root / system).exists()
    for system in systems
}

folder_status

In [ ]:
progress_rows = []

for system in systems:
    system_root = md_tests_root / system
    dry_topology = system_root / "gromacs" / "dry_polymer" / "topol.top"
    dry_topology_charge = read_primary_molecule_charge(dry_topology)
    solvated_dir = system_root / "gromacs" / "solvated_polymer"
    hpc_script = solvated_dir / "run_hpc_equilibration_production.slurm"
    expected_files = {
        "input_sdf": repo_root / "examples" / "output" / "polymer_structures" / f"{system}_R.sdf",
        "gaff2_prmtop": system_root / "gaff2" / f"{system}.prmtop",
        "gaff2_inpcrd": system_root / "gaff2" / f"{system}.inpcrd",
        "gaff2_mol2": system_root / "gaff2" / f"{system}.gaff2.mol2",
        "gaff2_frcmod": system_root / "gaff2" / f"{system}.gaff2.frcmod",
        "openmm_summary": system_root / "openmm" / "dry_polymer" / "openmm_summary.log",
        "openmm_final_pdb": system_root / "openmm" / "dry_polymer" / "final.pdb",
        "gromacs_dry_gro": system_root / "gromacs" / "dry_polymer" / "step5_input.gro",
        "gromacs_dry_top": system_root / "gromacs" / "dry_polymer" / "topol.top",
        "gromacs_system_solvated_gro": solvated_dir / "system_solvated.gro",
        "gromacs_ions_tpr": solvated_dir / "ions.tpr",
        "gromacs_system_neutralized_gro": solvated_dir / "system_neutralized.gro",
        "gromacs_solvated_gro": solvated_dir / "step5_input.gro",
        "gromacs_solvated_top": solvated_dir / "topol.top",
        "gromacs_solvate_script": solvated_dir / "run_solvate_local.sh",
        "gromacs_local_minimization_script": solvated_dir / "run_step6_local.sh",
        "gromacs_minimization_tpr": solvated_dir / "step6.0_minimization.tpr",
        "gromacs_minimized_gro": solvated_dir / "step6.0_minimization.gro",
        "gromacs_ions_mdp": solvated_dir / "ions.mdp",
        "gromacs_tip3p": solvated_dir / "TIP3_SOL.itp",
        "gromacs_sodium": solvated_dir / "SOD.itp",
        "gromacs_chloride": solvated_dir / "CLA.itp",
        "gromacs_nvt_mdp": solvated_dir / "step6.1_nvt.mdp",
        "gromacs_npt_mdp": solvated_dir / "step6.2_npt.mdp",
        "gromacs_production_mdp": solvated_dir / "step7_production.mdp",
        "gromacs_hpc_script": hpc_script,
    }
    progress_rows.append({
        "system": system,
        **{label: path.exists() for label, path in expected_files.items()},
        "gromacs_zero_net_topology_charge": dry_topology_charge is not None and abs(dry_topology_charge) <= 1e-4,
        "gromacs_local_minimization_maxwarn_1": (solvated_dir / "run_step6_local.sh").exists() and "step6.0_minimization.mdp" in (solvated_dir / "run_step6_local.sh").read_text() and "-maxwarn 1" in (solvated_dir / "run_step6_local.sh").read_text(),
        "gromacs_hpc_nvt_maxwarn_1": hpc_script.exists() and "step6.1_nvt.mdp" in hpc_script.read_text() and "-maxwarn 1" in hpc_script.read_text(),
    })

progress_rows

In [ ]:
for row in progress_rows:
    completed = sum(value is True for key, value in row.items() if key != "system")
    total = len(row) - 1
    missing = [key for key, value in row.items() if key != "system" and value is not True]
    print(f"{row['system']}: {completed}/{total} benchmark checks passed")
    if missing:
        print("  Missing or not yet passed:", ", ".join(missing))

## Selected Polymer Status

This final cell confirms whether the selected polymer is prepared for HPC. Submission is handled automatically by the run cell when `sbatch` is available.

In [ ]:
selected_solvated_dir = md_tests_root / selected_system / "gromacs" / "solvated_polymer"
minimized_structure = selected_solvated_dir / "step6.0_minimization.gro"
hpc_script = selected_solvated_dir / "run_hpc_equilibration_production.slurm"

print(f"Selected polymer: {selected_system}")
print(f"Minimized structure exists: {minimized_structure.exists()}")
print(f"HPC script exists: {hpc_script.exists()}")
print(f"Submission recorded: {(selected_solvated_dir / 'hpc_submission.txt').exists()}")